# 18. 3D geometry and rendering — NeRF and 3D Gaussian Splatting

This notebook keeps the rendering equations while reducing only scene size and image resolution. The 3DGS section now uses the original parameter types: **3D position, three log-scales, a normalized quaternion, opacity logit, and spherical-harmonic coefficients**. The previous single z-axis angle is removed.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(3)
device = torch.device("cpu")
print("device:", device)


## 1. NeRF ray volume rendering


In [ ]:
depths = torch.linspace(0.5, 3.0, 6, device=device)
density = torch.tensor(
    [0.2, 0.5, 1.0, 0.3, 0.1, 0.05],
    device=device,
)
colors = torch.rand(6, 3, device=device)

delta = torch.diff(depths)
delta = torch.cat([delta, delta[-1:]])
alpha = 1 - torch.exp(-density * delta)

survival = torch.cat(
    [torch.ones(1, device=device), 1 - alpha + 1e-8]
)
transmittance = torch.cumprod(survival, dim=0)[:-1]
weights = transmittance * alpha
pixel_color = (weights[:, None] * colors).sum(dim=0)

print("NeRF color:", pixel_color)


## 2. Quaternion rotation and anisotropic 3D covariance

3DGS parameterizes scale in three axes and a full 3D rotation. The covariance is formed as `Sigma = L L^T` with `L = R(q) diag(exp(s))`.


In [ ]:
def quaternion_to_rotation(quaternion):
    quaternion = F.normalize(quaternion, dim=-1)
    w, x, y, z = quaternion.unbind(dim=-1)

    rotation = torch.stack(
        [
            1 - 2 * (y.square() + z.square()),
            2 * (x * y - w * z),
            2 * (x * z + w * y),
            2 * (x * y + w * z),
            1 - 2 * (x.square() + z.square()),
            2 * (y * z - w * x),
            2 * (x * z - w * y),
            2 * (y * z + w * x),
            1 - 2 * (x.square() + y.square()),
        ],
        dim=-1,
    )
    return rotation.view(-1, 3, 3)


def covariance_3d(log_scales, quaternion):
    scales = torch.exp(log_scales)
    rotation = quaternion_to_rotation(quaternion)
    linear = rotation @ torch.diag_embed(scales)
    return linear @ linear.transpose(-1, -2)


## 3. Perspective projection and screen covariance


In [ ]:
def project_gaussians(
    means_3d,
    covariance_3d_values,
    height,
    width,
    fx=20.0,
    fy=20.0,
):
    z = means_3d[:, 2].clamp_min(0.2)
    u = fx * means_3d[:, 0] / z + width / 2
    v = fy * means_3d[:, 1] / z + height / 2
    means_2d = torch.stack([u, v], dim=-1)

    projected_covariances = []
    for index in range(means_3d.size(0)):
        x, y, depth = means_3d[index]
        zero = torch.zeros((), device=means_3d.device)

        jacobian = torch.stack(
            [
                torch.stack([fx / depth, zero, -fx * x / depth.square()]),
                torch.stack([zero, fy / depth, -fy * y / depth.square()]),
            ]
        )
        covariance_2d = (
            jacobian
            @ covariance_3d_values[index]
            @ jacobian.transpose(0, 1)
        )
        covariance_2d = covariance_2d + 1e-4 * torch.eye(
            2,
            device=means_3d.device,
        )
        projected_covariances.append(covariance_2d)

    return means_2d, torch.stack(projected_covariances), z


## 4. Degree-1 spherical harmonics

The original method supports higher SH degree during training; degree 1 is used here only to reduce the coefficient tensor size. The directional SH path itself is retained.


In [ ]:
def spherical_harmonics_degree1(direction):
    direction = F.normalize(direction, dim=-1)
    x, y, z = direction.unbind(dim=-1)

    c0 = 0.2820947918
    c1 = 0.4886025119
    return torch.stack(
        [
            torch.full_like(x, c0),
            -c1 * y,
            c1 * z,
            -c1 * x,
        ],
        dim=-1,
    )


def evaluate_spherical_harmonics(coefficients, direction):
    basis = spherical_harmonics_degree1(direction)
    rgb = torch.einsum(
        "nk,nkc->nc",
        basis,
        coefficients,
    )
    return torch.sigmoid(rgb)


## 5. Differentiable small 3D Gaussian rasterizer

The implementation projects every anisotropic Gaussian, evaluates its elliptical footprint on a small pixel grid, sorts primitives by camera depth, and performs front-to-back alpha compositing.


In [ ]:
def render_gaussians(
    means_3d,
    log_scales,
    quaternion,
    opacity_logits,
    sh_coefficients,
    height=8,
    width=8,
):
    covariances_3d = covariance_3d(log_scales, quaternion)
    means_2d, covariances_2d, depth = project_gaussians(
        means_3d,
        covariances_3d,
        height,
        width,
    )

    y_grid, x_grid = torch.meshgrid(
        torch.arange(height, device=means_3d.device, dtype=means_3d.dtype),
        torch.arange(width, device=means_3d.device, dtype=means_3d.dtype),
        indexing="ij",
    )
    pixels = torch.stack([x_grid, y_grid], dim=-1)

    colors = evaluate_spherical_harmonics(
        sh_coefficients,
        -means_3d,
    )
    opacities = torch.sigmoid(opacity_logits).squeeze(-1)

    image = torch.zeros(
        height,
        width,
        3,
        device=means_3d.device,
    )
    transmittance = torch.ones(
        height,
        width,
        device=means_3d.device,
    )

    for gaussian_index in depth.argsort():
        offset = pixels - means_2d[gaussian_index]
        inverse_covariance = torch.linalg.inv(
            covariances_2d[gaussian_index]
        )
        mahalanobis = torch.einsum(
            "...i,ij,...j->...",
            offset,
            inverse_covariance,
            offset,
        )
        gaussian = torch.exp(-0.5 * mahalanobis)
        alpha = (
            opacities[gaussian_index] * gaussian
        ).clamp(0.0, 0.99)

        weight = transmittance * alpha
        image = image + weight[..., None] * colors[gaussian_index]
        transmittance = transmittance * (1 - alpha)

    return image


## 6. Five-step CPU optimization check

A tiny target image is rendered from fixed Gaussian parameters. A perturbed copy then optimizes the same 3DGS parameters for five steps. This verifies gradients through quaternion rotation, covariance projection, SH color, opacity, and compositing.


In [ ]:
true_means = torch.tensor(
    [
        [-0.12, -0.08, 1.8],
        [0.15, -0.05, 2.1],
        [-0.05, 0.15, 2.4],
    ],
    device=device,
)
true_log_scales = torch.log(
    torch.tensor(
        [
            [0.12, 0.08, 0.10],
            [0.09, 0.11, 0.07],
            [0.10, 0.07, 0.12],
        ],
        device=device,
    )
)
true_quaternion = F.normalize(torch.randn(3, 4, device=device), dim=-1)
true_opacity_logits = torch.tensor(
    [[1.0], [0.6], [0.8]],
    device=device,
)
true_sh = torch.randn(3, 4, 3, device=device) * 0.3

with torch.no_grad():
    target_image = render_gaussians(
        true_means,
        true_log_scales,
        true_quaternion,
        true_opacity_logits,
        true_sh,
    )

means = nn.Parameter(
    true_means + 0.03 * torch.randn_like(true_means)
)
log_scales = nn.Parameter(true_log_scales.clone())
quaternion = nn.Parameter(
    true_quaternion + 0.05 * torch.randn_like(true_quaternion)
)
opacity_logits = nn.Parameter(true_opacity_logits.clone())
sh_coefficients = nn.Parameter(
    true_sh + 0.05 * torch.randn_like(true_sh)
)

optimizer = torch.optim.Adam(
    [
        means,
        log_scales,
        quaternion,
        opacity_logits,
        sh_coefficients,
    ],
    lr=5e-3,
)

loss_history = []
for step in range(5):
    optimizer.zero_grad()
    rendered = render_gaussians(
        means,
        log_scales,
        quaternion,
        opacity_logits,
        sh_coefficients,
    )
    loss = F.mse_loss(rendered, target_image)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    print(f"step {step + 1}: loss={loss.item():.8f}")

print("loss history:", loss_history)


## References and provenance

- NeRF: density-to-alpha, transmittance, and weighted color integration.
- 3D Gaussian Splatting: exponential anisotropic scales, normalized quaternion rotation, covariance projection with the perspective Jacobian, spherical-harmonic appearance, sigmoid opacity, depth ordering, and front-to-back alpha compositing.

Only Gaussian count, image resolution, and SH coefficient count are reduced.
